# Initialization

In [ ]:
import os
import sys
from tqdm import tqdm
import evaluate
import statistics

# Add root folder into paths
sys.path.append(os.path.abspath('..'))

# Now we can import our own classes/functions
from utils.datahandling import load_database
from utils.qna_irm import qna_irm_initialize, qna_irm_pipeline

In [ ]:
THRESHOLD = 0.2
RELATED_THRESHOLD = 0.5
K=1
sklearn_model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
data_json_path = "../datasets/combined_dataset.json" # local dataset json
local_generator_path = "../saved_models/meltemi 2025-07-04 08:46:26" # local fine-tuned model folder
instructions_path = '../instructions/v1 LLM instructions.txt'
unrelated_response_path = '../instructions/v1 LLM unrelated response.txt'

# Read the instructions that will be given to the model
with open(instructions_path, 'r', encoding='utf-8') as file:
    system_instructions = file.read()

# Read the response the model gives if the question is unrelated
with open(unrelated_response_path, 'r', encoding='utf-8') as file:
    unrelated_response = file.read()

In [ ]:
# Initialize irm variables
index, model, corpus, answers = qna_irm_initialize(data_json_path,
                                                              sklearn_model_name,
                                                              local_generator_path)

# Scoring

In [ ]:
# Load the rephrased data
data = load_database('../datasets/rephrased_faq_v1.json')
data.extend(load_database('../datasets/rephrased_faq_v2.json'))
data.extend(load_database('../datasets/rephrased_faq_v3.json'))

exact_match = 0
irm_predictions = []


# Use the irm pipeline for each input, but keep stats and show results in the meantime
for datum in tqdm(data):
    prediction, is_retrieved, is_related = qna_irm_pipeline(
        datum['input'],
        unrelated_response,
        index,
        model,
        answers,
        K,
        THRESHOLD,
        RELATED_THRESHOLD,
    )
                                                

    if is_retrieved:
        exact_match += 1
        irm_predictions.append(prediction)

print('Exact Match:', 100*exact_match/len(data))

In [ ]:
# Load metrics
bleu =  evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# bleurt = evaluate.load("bleurt", module_type="metric")
# exact_match = evaluate.load("exact_match")

outputs = [datum['output'] for datum in data]

In [ ]:
# BLEU expects list-of-lists for references
bleu_score = bleu.compute(predictions=irm_predictions, references=outputs)
print("BLEU:", bleu_score)

# ROUGE and METEOR expect plain list of strings
rouge_score = rouge.compute(predictions=irm_predictions, references=outputs)
print("ROUGE:", rouge_score)

meteor_score = meteor.compute(predictions=irm_predictions, references=outputs)
print("METEOR:", meteor_score)

# exact_match_score = exact_match.compute(predictions=predictions, references=outputs)
# print("EXACT MATCH:", exact_match_score)

In [ ]:
bertscore_score = bertscore.compute(predictions=irm_predictions, references=outputs, lang="gr", device="cpu")
print("BERTSCORE AVERAGE PRECISION:", statistics.mean(bertscore_score['precision']))
print("BERTSCORE AVERAGE RECALL:", statistics.mean(bertscore_score['recall']))
print("BERTSCORE AVERAGE F1:", statistics.mean(bertscore_score['f1']))

# Playground

In [ ]:
while True:
    print("\nΕσείς:", end=" ")
    user_input = input()
    
    # Handle special commands
    if user_input.lower() in ['έξοδος', 'exit', 'quit']:
        break

    print(user_input)
    
    # Generate and display response
    answer, retrieved, related = qna_irm_pipeline(
        user_input,
        unrelated_response,
        index,
        model,
        answers,
        K,
        THRESHOLD,
        RELATED_THRESHOLD)

    print(retrieved, 'OpsyedAI:', answer, flush=True)
    print()